In [ ]:
from tvDatafeed import TvDatafeed, Interval
import pandas as pd
import numpy as np
from datetime import datetime

tv = TvDatafeed()

CONTRATOS = ["DI1G2026", "DI1H2026", "DI1J2026", "DI1K2026"]

VENCIMENTOS = {
    "DI1G2026": pd.Timestamp("2026-02-02"),
    "DI1H2026": pd.Timestamp("2026-03-02"),
    "DI1J2026": pd.Timestamp("2026-04-01"),
    "DI1K2026": pd.Timestamp("2026-05-04"),
}

PERIODO_INICIO = pd.Timestamp("2026-01-02")
PERIODO_FIM    = pd.Timestamp("2026-04-30")

B3_FERIADOS = [
    "2026-01-01", "2026-02-16", "2026-02-17",
    "2026-04-03", "2026-04-21", "2026-05-01",
]

# 1. Coleta de cada contrato
print("Coletando contratos do TradingView...\n")
dados_brutos = []
for symbol in CONTRATOS:
    print(f"  {symbol}...", end=" ", flush=True)
    df = tv.get_hist(symbol=symbol, exchange="BMFBOVESPA",
                     interval=Interval.in_daily, n_bars=300)
    if df is None or len(df) == 0:
        print("VAZIO")
        continue
    df = df.reset_index()
    df["Data"] = pd.to_datetime(df["datetime"]).dt.normalize()
    df = df[(df["Data"] >= PERIODO_INICIO) & (df["Data"] <= PERIODO_FIM)].copy()
    df["Contrato"] = symbol
    df["Taxa"] = df["close"]
    dados_brutos.append(df[["Data", "Contrato", "Taxa"]])
    print(f"{len(df)} dias no periodo de teste")

if not dados_brutos:
    raise SystemExit("Nenhum dado retornado.")

dados = pd.concat(dados_brutos, ignore_index=True)

# 2. Reindex para grade completa de dias uteis B3 + forward-fill
todos_dias = pd.bdate_range(
    start=PERIODO_INICIO, end=PERIODO_FIM,
    freq="C", holidays=B3_FERIADOS,
)

preenchido = []
for symbol in CONTRATOS:
    serie = (
        dados[dados["Contrato"] == symbol]
        .set_index("Data")
        .sort_index()
        .reindex(todos_dias)
    )
    serie["Contrato"] = symbol
    serie["Taxa"] = serie["Taxa"].ffill()
    preenchido.append(serie.reset_index().rename(columns={"index": "Data"}))

resultado = (
    pd.concat(preenchido, ignore_index=True)
    .sort_values(["Data", "Contrato"])
    .reset_index(drop=True)
)

# 3. Calcula DU e PU (PU back-calculado pela formula inversa)
def du_b3(data, venc):
    return int(np.busday_count(
        np.datetime64(data.date(), "D"),
        np.datetime64(venc.date(), "D"),
        holidays=np.array(B3_FERIADOS, dtype="datetime64[D]"),
    ))

resultado["DU"] = resultado.apply(
    lambda r: du_b3(r["Data"], VENCIMENTOS[r["Contrato"]]), axis=1
)
resultado["PU"] = 100000 / (1 + resultado["Taxa"]/100) ** (resultado["DU"]/252)

# 4. Salva XLSX com formatacao
saida = resultado[["Data", "Contrato", "PU", "Taxa"]].copy()
saida.columns = ["Data", "Contrato", "PU", "Taxa_a.a."]
saida["PU"]        = saida["PU"].round(2)
saida["Taxa_a.a."] = (saida["Taxa_a.a."] / 100).round(6)  # como decimal para Excel formatar como %

with pd.ExcelWriter("dados/dados_modelo_tcc_di_futuro.xlsx", engine="openpyxl") as writer:
    saida.to_excel(writer, sheet_name="DI_Futuro", index=False)
    ws = writer.sheets["DI_Futuro"]
    # Formato das celulas
    for cell in ws["A"][1:]: cell.number_format = "DD/MM/YYYY"   # Data
    for cell in ws["C"][1:]: cell.number_format = "#,##0.00"     # PU
    for cell in ws["D"][1:]: cell.number_format = "0.0000%"      # Taxa
    # Largura das colunas
    ws.column_dimensions["A"].width = 12
    ws.column_dimensions["B"].width = 12
    ws.column_dimensions["C"].width = 14
    ws.column_dimensions["D"].width = 14

print(f"\nXLSX salvo: di_futuro.xlsx ({len(saida)} linhas, esperado ~244)")
print("\nPrimeiras 8 linhas:")
print(saida.head(8).to_string(index=False))
print("\nUltimas 8 linhas:")
print(saida.tail(8).to_string(index=False))
